# Can we trust the confidence score?

Every label the model produces comes with a **confidence**. If that number were
trustworthy, wrong labels would carry low confidence and we could catch mistakes
automatically.

One step per cell below, each showing its own code and result.


In [1]:
import os
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

import json
import pandas as pd
from IPython.display import display

from dmpbridge.core import paths as P
from dmpbridge.evaluation.evaluate import (
    containment, extract_gold, resolve_old_gt_path, tokenize,
)

MODELS = ['llama3.1:8b', 'gemma4:e4b', 'llama3.3:70b']
EXTRACTOR = 'pdfplumber'
SAMPLES = range(1, 11)

pd.set_option('display.max_colwidth', 60)


## Step 1 — Load the labeled JSON

This is what the model produced: one entry per **line** of the PDF, each with a
label and a confidence.


In [2]:
path = P.labeled_path(P.make_tag('llama3.1:8b', EXTRACTOR), 1)
print(path)

lines = json.loads(path.read_text(encoding='utf-8'))
print(f'{len(lines)} lines in this file')


C:\Users\Nahid\dmpbridge\data\output\2_labeled\llama3.1-8b_pdfplumber_whole_doc\sample1.json
78 lines in this file


In [3]:
# The first four lines, showing only the fields this notebook uses.
display(pd.DataFrame(lines)[['text', 'label', 'confidence']].head(4))


,text,label,confidence
0,DATA MANAGEMENT AND SHARING PLAN,title,1.0
1,Element 1: Data Type:,section.title,1.0
2,A. Types and amount of scientific data expected to be ge...,question.text,1.0
3,This secondary data analysis project will analyze deiden...,answer.text,1.0


## Step 2 — Join the lines back into items

A paragraph in the PDF is several lines, but the annotation records it as **one
item**. So neighbouring lines carrying the **same label** are put back together.

This is what the pipeline already does when it builds its final output.


In [4]:
def join_lines(lines):
    """Neighbouring lines with the same label become one item."""
    items = []
    current = None
    for line in lines:
        label = line['label']
        confidence = float(line['confidence'])
        if current is not None and current['label'] == label:
            current['text'] += ' ' + line['text']
            current['line_confidences'].append(confidence)
        else:
            if current is not None:
                items.append(current)
            current = {'label': label,
                       'text': line['text'],
                       'line_confidences': [confidence]}
    if current is not None:
        items.append(current)
    return items


In [5]:
items = join_lines(lines)
print(f'{len(lines)} lines  ->  {len(items)} items')


78 lines  ->  30 items


### See it happen on one paragraph

Lines 3, 4 and 5 of the file all carry the same label, so they become a single item.


In [6]:
print('BEFORE — three separate lines:')
for line in lines[3:6]:
    print(f"  [{line['label']}]  conf {line['confidence']}  {line['text'][:58]!r}")

print()
print('AFTER — one item:')
joined = join_lines(lines[3:6])[0]
print(f"  [{joined['label']}]  from {len(joined['line_confidences'])} lines")
print(f"  {joined['text'][:110]!r}")


BEFORE — three separate lines:
  [answer.text]  conf 1.0  'This secondary data analysis project will analyze deidenti'
  [answer.text]  conf 1.0  'and the publicly available NHANES cohorts (wrist NHANES 20'
  [answer.text]  conf 1.0  'The studies include (i) the RISE Study, (ii) the SOL-VIDA '

AFTER — one item:
  [answer.text]  from 3 lines
  'This secondary data analysis project will analyze deidentified data from 48,218 participants from eight studie'


## Step 3 — Give each item one confidence

An item is built from several lines, each with its own confidence. We take the
**lowest** — a paragraph is only as trustworthy as its shakiest line.


In [7]:
for item in items:
    item['confidence'] = min(item['line_confidences'])

# The item built from the most lines, to show the rule working.
biggest = max(items, key=lambda it: len(it['line_confidences']))
print(f"an item made of {len(biggest['line_confidences'])} lines")
print(f"  line confidences : {biggest['line_confidences']}")
print(f"  lowest           : {biggest['confidence']}  <- the item's confidence")


an item made of 12 lines
  line confidences : [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
  lowest           : 1.0  <- the item's confidence


## Step 4 — Check each item against the annotation

The annotation says what is really in the document. Each item is matched to the
paragraph it belongs to, then we check whether the label agrees.


In [8]:
def check(item, annotation):
    """Return the item's true label, or None if it matches no paragraph."""
    words = tokenize(item['text'])
    if len(words) < 3:
        return None                     # too short to place reliably
    best_score, best_label = 0.0, None
    for gold_text, gold_label in annotation:
        score = containment(words, tokenize(gold_text))
        if score > best_score:
            best_score, best_label = score, gold_label
    return best_label if best_score >= 0.75 else None


In [9]:
annotation = extract_gold(resolve_old_gt_path(1))
print(f'the annotation for sample 1 has {len(annotation)} items\n')

for item in items[:6]:
    true_label = check(item, annotation)
    if true_label is None:
        print(f"  (skipped — no match)  {item['text'][:52]!r}")
    else:
        verdict = 'correct' if item['label'] == true_label else 'WRONG'
        print(f"  conf {item['confidence']:.2f}  said {item['label']:<20}"
              f"true {true_label:<20}{verdict}")


the annotation for sample 1 has 28 items

  conf 1.00  said title               true title               correct
  conf 1.00  said section.title       true section.title       correct
  conf 1.00  said question.text       true question.text       correct
  conf 1.00  said answer.text         true answer.text         correct
  conf 1.00  said section.title       true question.text       WRONG
  conf 1.00  said answer.text         true answer.text         correct


## Step 5 — Do all of that for every model and every document

The same four steps, in a loop: load, join, take the lowest confidence, check.


In [10]:
rows = []
for sample in SAMPLES:
    annotation = extract_gold(resolve_old_gt_path(sample))
    for model in MODELS:
        path = P.labeled_path(P.make_tag(model, EXTRACTOR), sample)
        lines = json.loads(path.read_text(encoding='utf-8'))
        for item in join_lines(lines):
            item['confidence'] = min(item['line_confidences'])
            true_label = check(item, annotation)
            if true_label is None:
                continue
            rows.append({'model': model,
                         'sample': sample,
                         'confidence': item['confidence'],
                         'model said': item['label'],
                         'true label': true_label,
                         'correct': item['label'] == true_label,
                         'text': item['text']})

df = pd.DataFrame(rows)
print(f'{len(df)} items checked')
display(df.head(5))


768 items checked


,model,sample,confidence,model said,true label,correct,text
0,llama3.1:8b,1,1.0,title,title,True,DATA MANAGEMENT AND SHARING PLAN
1,llama3.1:8b,1,1.0,section.title,section.title,True,Element 1: Data Type:
2,llama3.1:8b,1,1.0,question.text,question.text,True,A. Types and amount of scientific data expected to be ge...
3,llama3.1:8b,1,1.0,answer.text,answer.text,True,This secondary data analysis project will analyze deiden...
4,llama3.1:8b,1,1.0,section.title,question.text,False,"B. Scientific data that will be preserved and shared, an..."


## Step 6 — The answer

If confidence were trustworthy, the wrong items would be clearly less confident
than the right ones.


In [11]:
answer = (df.groupby('model')
          .apply(lambda d: pd.Series({
              'items': len(d),
              'correct': d['correct'].sum(),
              'accuracy': d['correct'].mean(),
              'confidence when RIGHT': d[d['correct']]['confidence'].mean(),
              'confidence when WRONG': d[~d['correct']]['confidence'].mean(),
          }), include_groups=False)
          .reindex(MODELS))
display(answer.style.format({'items': '{:.0f}', 'correct': '{:.0f}',
                             'accuracy': '{:.0%}',
                             'confidence when RIGHT': '{:.2f}',
                             'confidence when WRONG': '{:.2f}'}))


,items,correct,accuracy,confidence when RIGHT,confidence when WRONG
model,,,,,
llama3.1:8b,436,214,49%,0.89,0.89
gemma4:e4b,155,117,75%,0.96,0.91
llama3.3:70b,177,133,75%,0.88,0.83


### The wrong items that were most confident


In [12]:
worst = (df[~df['correct']].sort_values('confidence', ascending=False)
         [['model', 'confidence', 'true label', 'model said', 'text']]
         .head(8).reset_index(drop=True))
display(worst.style.background_gradient(cmap='Reds', subset=['confidence'],
                                        vmin=0.5, vmax=1.0)
        .format({'confidence': '{:.2f}'}))


,model,confidence,true label,model said,text
0,llama3.1:8b,1.00,question.text,section.title,"B. Scientific data that will be preserved and shared, and the rationale for doing so:"
1,llama3.1:8b,1.00,answer.text,question.text,The following data will be created as a result of this project:
2,llama3.1:8b,1.00,question.text,section.title,"C. Metadata, other relevant data, and associated documentation:"
3,llama3.1:8b,1.00,question.text,section.title,B. How scientific data will be findable and identifiable:
4,llama3.1:8b,1.00,question.text,section.title,C. When and how long the scientific data will be made available:
5,llama3.1:8b,1.00,answer.text,question.text,about using their data:
6,llama3.1:8b,1.00,question.text,section.title,"Element 5: Access, Distribution, or Reuse Considerations: A. Factors affecting subsequent access, distribution, or reuse of scientific data:"
7,llama3.3:70b,1.00,answer.text,section.description,The following data will be created as a result of this project:


## Conclusion

**The confidence score cannot be used to catch mistakes.**

The two confidence columns in step 6 are close together for every model — wrong
items are about as confident as right ones. The table above shows mistakes claimed
at complete certainty.

So a high confidence does not mean a correct label, and it must not be used to
accept one automatically.
